# 🇻🇳 RAG Pipeline — UIT-ViQuAD 2.0

**Mục tiêu notebook này:**
1. Load dataset `uitnlp/ViQuAD2.0` từ Hugging Face
2. Chunking & làm sạch dữ liệu
3. Encode toàn bộ context bằng PhoBERT bi-encoder (GPU)
4. Build FAISS index
5. Evaluate Recall@K
6. Lưu artifacts vào local path (project folder)

**Runtime:** Colab Local Runtime (GPU của máy bạn)

---
> ℹ️ Notebook này chạy trên **Colab Local Runtime** — không cần Google Drive,
> artifacts được lưu thẳng vào thư mục project trên máy bạn.

In [59]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [60]:
import os

# ─── CHỈNH Ở ĐÂY ───────────────────────────────────────────────────────────
# Thư mục gốc project trên máy bạn (dùng đường dẫn tuyệt đối)
PROJECT_ROOT = "/content/drive/MyDrive/Code/Colab"
# ────────────────────────────────────────────────────────────────────────────

# Các sub-folder — không cần chỉnh
DATA_DIR     = os.path.join(PROJECT_ROOT, 'data')
NOTEBOOK_DIR = os.path.join(PROJECT_ROOT, 'notebooks')
SRC_DIR      = os.path.join(PROJECT_ROOT, 'src')
HF_CACHE     = os.path.join(PROJECT_ROOT, '.hf_cache')    # cache dataset HF
MODEL_CACHE  = os.path.join(PROJECT_ROOT, '.model_cache') # cache model weights

for d in [DATA_DIR, NOTEBOOK_DIR, SRC_DIR, HF_CACHE, MODEL_CACHE]:
    os.makedirs(d, exist_ok=True)

print('📂 Project structure:')
print(f'   ROOT         : {PROJECT_ROOT}')
print(f'   DATA_DIR     : {DATA_DIR}')
print(f'   HF_CACHE     : {HF_CACHE}')
print(f'   MODEL_CACHE  : {MODEL_CACHE}')
print()
print('✅ Directories ready')

📂 Project structure:
   ROOT         : /content/drive/MyDrive/Code/Colab
   DATA_DIR     : /content/drive/MyDrive/Code/Colab/data
   HF_CACHE     : /content/drive/MyDrive/Code/Colab/.hf_cache
   MODEL_CACHE  : /content/drive/MyDrive/Code/Colab/.model_cache

✅ Directories ready


## 📦 Cell 1 — Cài đặt thư viện

In [3]:
import subprocess, sys

# ── Cài faiss-gpu đúng cách cho CUDA 12.x ──────────────────────────────────
# PyPI faiss-gpu không support CUDA 12+, phải cài qua conda hoặc pip wheel
try:
    import faiss
    res = faiss.StandardGpuResources()
    print(f'✅ faiss-gpu already working')
except Exception:
    print('⏳ Installing faiss-gpu (CUDA 12 compatible)...')
    # Thử pip wheel build mới nhất hỗ trợ CUDA 12
    ret = subprocess.call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'faiss-gpu-cu12',   # wheel riêng cho CUDA 12.x
    ])
    if ret != 0:
        # Fallback: cài faiss-cpu, vẫn chạy được, chỉ search chậm hơn
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu'
        ])
        print('⚠️  Dùng faiss-cpu — search vẫn OK, chỉ chậm hơn GPU ~3x')
    else:
        print('✅ faiss-gpu-cu12 installed')

# ── Các thư viện còn lại ────────────────────────────────────────────────────
other_pkgs = [
    'datasets', 'sentence-transformers', 'transformers',
    'torch', 'tqdm', 'numpy', 'pandas', 'accelerate',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + other_pkgs)
print('✅ Other packages installed')

# ── Kiểm tra GPU ─────────────────────────────────────────────────────────────
import torch, faiss
print(f'\ntorch  : {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Kiểm tra faiss có dùng được GPU không
try:
    res = faiss.StandardGpuResources()
    print(f'faiss  : GPU ✅')
    FAISS_GPU = True
except Exception as e:
    print(f'faiss  : CPU only (GPU init failed: {e})')
    FAISS_GPU = False

⏳ Installing faiss-gpu (CUDA 12 compatible)...
⚠️  Dùng faiss-cpu — search vẫn OK, chỉ chậm hơn GPU ~3x
✅ Other packages installed

torch  : 2.3.1+cpu
CUDA   : False
faiss  : CPU only (GPU init failed: module 'faiss' has no attribute 'StandardGpuResources')


## 📥 Cell 2 — Load dataset ViQuAD 2.0

In [4]:
from datasets import load_dataset

print('⏳ Loading dataset...')
dataset = load_dataset('taidng/UIT-ViQuAD2.0', cache_dir=HF_CACHE)

print('\n📊 Dataset overview:')
print(dataset)

print('\n📌 Sample record (train[0]):')
sample = dataset['train'][0]
for key, val in sample.items():
    s = str(val)
    print(f'  [{key:20s}]: {s[:130] + "..." if len(s) > 130 else s}')

⏳ Loading dataset...


Generating train split:   0%|          | 0/28454 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3814 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7301 [00:00<?, ? examples/s]


📊 Dataset overview:
DatasetDict({
    train: Dataset({
        features: ['id', 'uit_id', 'title', 'context', 'question', 'answers', 'is_impossible', 'plausible_answers'],
        num_rows: 28454
    })
    validation: Dataset({
        features: ['id', 'uit_id', 'title', 'context', 'question', 'answers', 'is_impossible', 'plausible_answers'],
        num_rows: 3814
    })
    test: Dataset({
        features: ['id', 'uit_id', 'title', 'context', 'question', 'answers', 'is_impossible', 'plausible_answers'],
        num_rows: 7301
    })
})

📌 Sample record (train[0]):
  [id                  ]: 0001-0001-0001
  [uit_id              ]: uit_000001
  [title               ]: Phạm Văn Đồng
  [context             ]: Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4 năm 2000) là Thủ tướng đầu tiên của nước Cộng hòa Xã hội chủ nghĩa Việt Nam từ năm ...
  [question            ]: Tên gọi nào được Phạm Văn Đồng sử dụng khi làm Phó chủ nhiệm cơ quan Biện sự xứ tại Quế Lâm?
  [answers             ]: {

## 🔧 Cell 3 — Chunking & Preprocessing

**Chiến lược cho ViQuAD2.0:**
- Mỗi `context` = 1 chunk (đã là đoạn văn hoàn chỉnh, không cần split thêm)
- Deduplicate: nhiều câu hỏi share cùng 1 context → chỉ index 1 lần
- Gắn `title` làm metadata để filter sau
- Tách riêng `is_impossible=True` để LLM reader biết trả lời "không có đáp án"

In [5]:
import re
from tqdm.auto import tqdm


def clean_text(text: str) -> str:
    """Làm sạch văn bản tiếng Việt cơ bản."""
    if not isinstance(text, str):
        return ''
    text = re.sub(r'\s+', ' ', text)                              # chuẩn hóa khoảng trắng
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text) # ký tự điều khiển
    return text.strip()


def build_corpus(dataset, split_name: str):
    """
    Trả về:
      chunks   — list[dict]: context duy nhất + metadata
      qa_pairs — list[dict]: tất cả QA pairs, trỏ đến chunk_id
    """
    seen_ctx = {}  # context_text -> chunk_id
    chunks   = []
    qa_pairs = []

    data = dataset[split_name]
    print(f'⏳ Processing {len(data):,} records từ split "{split_name}"...')

    for record in tqdm(data, desc=split_name):
        ctx_clean = clean_text(record.get('context', ''))
        if not ctx_clean:
            continue

        # Deduplicate theo context text
        if ctx_clean not in seen_ctx:
            chunk_id = len(chunks)
            seen_ctx[ctx_clean] = chunk_id
            chunks.append({
                'chunk_id': chunk_id,
                'title'   : clean_text(record.get('title', '')),
                'context' : ctx_clean,
                'char_len': len(ctx_clean),
            })
        else:
            chunk_id = seen_ctx[ctx_clean]

        qa_pairs.append({
            'id'               : str(record.get('id', record.get('uit_id', ''))),
            'chunk_id'         : chunk_id,
            'question'         : clean_text(record.get('question', '')),
            'answers'          : record.get('answers', {}),
            'is_impossible'    : bool(record.get('is_impossible', False)),
            'plausible_answers': record.get('plausible_answers', {}),
            'split'            : split_name,
        })

    return chunks, qa_pairs


# Build train
train_chunks, train_qa = build_corpus(dataset, 'train')

# Build validation nếu có
val_chunks, val_qa = [], []
if 'validation' in dataset:
    val_chunks, val_qa = build_corpus(dataset, 'validation')

# Merge + deduplicate cross-split
all_ctx_map = {c['context']: c for c in train_chunks}
for c in val_chunks:
    if c['context'] not in all_ctx_map:
        c['chunk_id'] = len(all_ctx_map)
        all_ctx_map[c['context']] = c

all_chunks = list(all_ctx_map.values())
for i, c in enumerate(all_chunks):  # re-index chunk_id liên tục
    c['chunk_id'] = i

# Thống kê
n_impossible = sum(1 for q in train_qa + val_qa if q['is_impossible'])
lengths = [c['char_len'] for c in all_chunks]

print(f'\n✅ Unique chunks (contexts): {len(all_chunks):,}')
print(f'   Train QA pairs          : {len(train_qa):,}')
print(f'   Val   QA pairs          : {len(val_qa):,}')
print(f'   is_impossible           : {n_impossible:,}')
print(f'\n📏 Context length (chars):')
print(f'   min={min(lengths)}, max={max(lengths)}, '
      f'mean={int(sum(lengths)/len(lengths))}, '
      f'median={sorted(lengths)[len(lengths)//2]}')

⏳ Processing 28,454 records từ split "train"...


train:   0%|          | 0/28454 [00:00<?, ?it/s]

⏳ Processing 3,814 records từ split "validation"...


validation:   0%|          | 0/3814 [00:00<?, ?it/s]


✅ Unique chunks (contexts): 4,658
   Train QA pairs          : 28,454
   Val   QA pairs          : 3,814
   is_impossible           : 10,377

📏 Context length (chars):
   min=465, max=7221, mean=821, median=731


In [6]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 3b — FIX: Remap chunk_id trong qa_pairs sau khi merge all_chunks
# Phải chạy ngay sau Cell 3, trước Cell 4
# ═══════════════════════════════════════════════════════════════════════════

# Sau khi Cell 3 re-index all_chunks, build lại lookup: context_text → chunk_id mới
ctx_to_new_id = {c['context']: c['chunk_id'] for c in all_chunks}

def remap_qa(qa_pairs, label=''):
    """Cập nhật chunk_id trong qa_pairs theo all_chunks mới nhất."""
    missing = 0
    for qa in qa_pairs:
        # Lấy context gốc từ all_chunks cũ (vẫn đúng vì chỉ id thay đổi, text không đổi)
        old_id = qa['chunk_id']
        # Tìm lại context text từ all_chunks theo old_id — không được dùng trực tiếp
        # vì old_id có thể đã lệch. Phải match qua context text.
        # → train_qa/val_qa không lưu context text → cần rebuild từ source
        pass
    return missing

# Cách đúng: rebuild chunk_id từ context text gốc trong dataset
print('🔧 Remapping chunk_id for train_qa...')
missing_train = 0
for qa in train_qa:
    # Lấy context text của câu hỏi này từ dataset gốc (đã có trong record khi build)
    # Vì qa_pairs không lưu context, cần dùng all_chunks[old_id].context
    # nhưng all_chunks đã bị re-index → old_id không tin được
    # → Fix thực sự: lưu context text vào qa_pairs ngay từ đầu
    pass

# ── Fix thực sự: lưu context_text trong qa_pairs ────────────────────────────
# Rebuild train_qa và val_qa với context text để remap được bất cứ lúc nào

print('🔧 Rebuilding qa_pairs với context_text...')

def build_corpus_fixed(dataset, split_name: str):
    seen_ctx = {}
    chunks   = []
    qa_pairs = []
    data     = dataset[split_name]

    for record in tqdm(data, desc=split_name):
        ctx_clean = clean_text(record.get('context', ''))
        if not ctx_clean:
            continue
        if ctx_clean not in seen_ctx:
            chunk_id = len(chunks)
            seen_ctx[ctx_clean] = chunk_id
            chunks.append({
                'chunk_id': chunk_id,
                'title'   : clean_text(record.get('title', '')),
                'context' : ctx_clean,
                'char_len': len(ctx_clean),
            })
        else:
            chunk_id = seen_ctx[ctx_clean]

        qa_pairs.append({
            'id'               : str(record.get('id', record.get('uit_id', ''))),
            'chunk_id'         : chunk_id,
            'context_text'     : ctx_clean,           # ← thêm dòng này
            'question'         : clean_text(record.get('question', '')),
            'answers'          : record.get('answers', {}),
            'is_impossible'    : bool(record.get('is_impossible', False)),
            'plausible_answers': record.get('plausible_answers', {}),
            'split'            : split_name,
        })
    return chunks, qa_pairs

train_chunks, train_qa = build_corpus_fixed(dataset, 'train')
val_chunks, val_qa     = [], []
if 'validation' in dataset:
    val_chunks, val_qa = build_corpus_fixed(dataset, 'validation')

# Merge + deduplicate (giống Cell 3)
all_ctx_map = {c['context']: c for c in train_chunks}
for c in val_chunks:
    if c['context'] not in all_ctx_map:
        c['chunk_id'] = len(all_ctx_map)
        all_ctx_map[c['context']] = c

all_chunks = list(all_ctx_map.values())
for i, c in enumerate(all_chunks):
    c['chunk_id'] = i

# Build lookup context_text → chunk_id cuối cùng
ctx_to_new_id = {c['context']: c['chunk_id'] for c in all_chunks}

# Remap chunk_id trong qa_pairs theo all_chunks mới
missing = 0
for qa in train_qa + val_qa:
    new_id = ctx_to_new_id.get(qa['context_text'])
    if new_id is None:
        missing += 1
        continue
    qa['chunk_id'] = new_id

print(f'✅ Remapped train_qa: {len(train_qa):,} pairs')
print(f'✅ Remapped val_qa  : {len(val_qa):,} pairs')
print(f'   Missing (không tìm được context): {missing}')

# Sanity check: lấy 3 cặp ngẫu nhiên, verify chunk_id đúng
import random
random.seed(0)
for qa in random.sample([q for q in train_qa if not q['is_impossible']], 3):
    chunk = all_chunks[qa['chunk_id']]
    match = chunk['context'] == qa['context_text']
    print(f'   chunk_id={qa["chunk_id"]}  context_match={match}  ✅' if match
          else f'   chunk_id={qa["chunk_id"]}  ❌ MISMATCH!')

🔧 Remapping chunk_id for train_qa...
🔧 Rebuilding qa_pairs với context_text...


train:   0%|          | 0/28454 [00:00<?, ?it/s]

validation:   0%|          | 0/3814 [00:00<?, ?it/s]

✅ Remapped train_qa: 28,454 pairs
✅ Remapped val_qa  : 3,814 pairs
   Missing (không tìm được context): 0
   chunk_id=2694  context_match=True  ✅
   chunk_id=2926  context_match=True  ✅
   chunk_id=290  context_match=True  ✅


## 🧠 Cell 4 — Load Embedding Model (PhoBERT Bi-encoder)

In [13]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 4 — Multi-Model Config & Pipeline Runner
# ═══════════════════════════════════════════════════════════════════════════
from sentence_transformers import SentenceTransformer
import torch, faiss, numpy as np, time, random
from tqdm.auto import tqdm

# ── Danh sách model muốn benchmark ─────────────────────────────────────────
# prefix: theo convention của từng model
#   - E5 family   → 'query: ' / 'passage: '
#   - PhoBERT/SBERT Vietnamese → prefix rỗng '' (model không dùng prefix)
MODEL_CONFIGS = [
    {
        'name'           : 'intfloat/multilingual-e5-base',
        'query_prefix'   : 'query: ',
        'passage_prefix' : 'passage: ',
        'note'           : 'Đa ngôn ngữ, backup tốt',
    },
    {
        'name'           : 'intfloat/multilingual-e5-small',
        'query_prefix'   : 'query: ',
        'passage_prefix' : 'passage: ',
        'note'           : 'Nhẹ hơn e5-base ~3x, nhanh hơn',
    },
    # {
    #     'name'           : 'keepitreal/vietnamese-sbert',
    #     'query_prefix'   : '',
    #     'passage_prefix' : '',
    #     'note'           : 'SBERT fine-tune tiếng Việt',
    # },
    # {
    #     'name'           : 'bkai-foundation-models/vietnamese-bi-encoder',
    #     'query_prefix'   : '',
    #     'passage_prefix' : '',
    #     'note'           : 'PhoBERT bi-encoder, recommended cho ViQuAD',
    # },
]

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Tự động chọn batch_size theo VRAM
if torch.cuda.is_available():
    vram_gb    = torch.cuda.get_device_properties(0).total_memory / 1e9
    BATCH_SIZE = 128 if vram_gb >= 8 else 64
    print(f'🖥️  GPU: {torch.cuda.get_device_name(0)}  ({vram_gb:.1f} GB VRAM)')
else:
    BATCH_SIZE = 16
    print(f'🖥️  CPU mode  (batch_size={BATCH_SIZE})')


# ── Hàm pipeline cho 1 model ────────────────────────────────────────────────
def run_model_pipeline(cfg: dict, chunks: list, qa_pairs: list,
                        k_list=(1, 3, 5, 10), eval_sample=500) -> dict:
    """
    Chạy toàn bộ pipeline cho 1 model config:
      load → encode corpus → build FAISS → evaluate Recall@K + MRR
    Trả về dict kết quả để so sánh.
    """
    model_name      = cfg['name']
    query_prefix    = cfg['query_prefix']
    passage_prefix  = cfg['passage_prefix']

    print(f'\n{"═"*68}')
    print(f'🔄  {model_name}')
    print(f'    note: {cfg["note"]}')
    print(f'{"═"*68}')

    # ── 1. Load model ────────────────────────────────────────────────────────
    model = SentenceTransformer(model_name, device=device, cache_folder=MODEL_CACHE)
    dim   = model.get_sentence_embedding_dimension()
    print(f'✅  Loaded  |  dim={dim}  max_seq={model.max_seq_length}')

    # Smoke test
    test = model.encode('Việt Nam là một quốc gia ở Đông Nam Á.',
                         normalize_embeddings=True)
    assert test.shape[0] == dim, 'Smoke test failed!'

    # ── 2. Encode corpus ─────────────────────────────────────────────────────
    texts = [passage_prefix + c['context'] for c in chunks]
    t0    = time.time()
    embs  = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    encode_time = time.time() - t0
    print(f'⚡  Encoded {len(texts):,} passages in {encode_time:.1f}s'
          f'  ({len(texts)/encode_time:.0f} p/s)  |  {embs.nbytes/1e6:.0f} MB')

    # ── 3. Build FAISS index ─────────────────────────────────────────────────
    if len(chunks) < 100_000:
        cpu_idx    = faiss.IndexFlatIP(dim)
        index_type = 'IndexFlatIP'
    else:
        nlist      = min(int(len(chunks) ** 0.5), 512)
        quantizer  = faiss.IndexFlatIP(dim)
        cpu_idx    = faiss.IndexIVFFlat(quantizer, dim, nlist,
                                         faiss.METRIC_INNER_PRODUCT)
        index_type = f'IndexIVFFlat(nlist={nlist})'
        cpu_idx.train(embs)
    cpu_idx.add(embs)

    # GPU transfer nếu có
    faiss_idx = cpu_idx
    if FAISS_GPU:
        try:
            res       = faiss.StandardGpuResources()
            faiss_idx = faiss.index_cpu_to_gpu(res, 0, cpu_idx)
            print(f'🗂️  FAISS {index_type} on GPU')
        except Exception as e:
            print(f'⚠️  FAISS GPU failed, dùng CPU: {e}')
    else:
        print(f'🗂️  FAISS {index_type} on CPU')

    # Sanity check: self-query phải = 1.0
    s, _ = faiss_idx.search(embs[0:1], 1)
    assert abs(s[0][0] - 1.0) < 1e-4, f'Self-query score {s[0][0]:.6f} ≠ 1.0'

    # ── 4. Evaluate Recall@K + MRR ───────────────────────────────────────────
    pool = [q for q in qa_pairs if not q['is_impossible']]
    random.seed(42)
    if len(pool) > eval_sample:
        pool = random.sample(pool, eval_sample)

    max_k  = max(k_list)
    hits   = {k: 0 for k in k_list}
    rr_sum = 0.0

    for qa in tqdm(pool, desc=f'Eval {model_name.split("/")[-1]}'):
        gt_id = qa['chunk_id']
        q_vec = model.encode(
            [query_prefix + qa['question']],
            normalize_embeddings=True,
            convert_to_numpy=True,
        ).astype(np.float32)

        _, idxs   = faiss_idx.search(q_vec, max_k)
        retrieved = [int(i) for i in idxs[0] if i != -1]

        for k in k_list:
            if gt_id in retrieved[:k]:
                hits[k] += 1

        # MRR
        for rank, idx in enumerate(retrieved):
            if idx == gt_id:
                rr_sum += 1.0 / (rank + 1)
                break

    n   = len(pool)
    mrr = rr_sum / n

    print(f'\n📊 Results — {model_name}')
    print(f'   Sample: {n:,} pairs')
    print(f'   MRR@{max_k}  : {mrr:.4f}  ({mrr*100:.1f}%)')
    for k in k_list:
        r   = hits[k] / n * 100
        bar = '█' * int(r / 5) + '░' * (20 - int(r / 5))
        tip = '✅' if r >= 85 else ('⚠️' if r >= 70 else '❌')
        print(f'   Recall@{k:<3}: {r:5.1f}%  |{bar}| {tip}')

    # Giải phóng GPU memory
    del model, embs, faiss_idx
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        'model'        : model_name,
        'note'         : cfg['note'],
        'dim'          : dim,
        'index_type'   : index_type,
        'encode_time_s': round(encode_time, 1),
        'mrr'          : round(mrr, 4),
        **{f'recall@{k}': round(hits[k] / n, 4) for k in k_list},
    }


print(f'\n📋 {len(MODEL_CONFIGS)} models sẽ được benchmark:')
for i, cfg in enumerate(MODEL_CONFIGS):
    print(f'   {i+1}. {cfg["name"]}  —  {cfg["note"]}')
print(f'\n▶  Gọi run_all_benchmarks() ở cell tiếp theo để chạy.')

🖥️  CPU mode  (batch_size=16)

📋 2 models sẽ được benchmark:
   1. intfloat/multilingual-e5-base  —  Đa ngôn ngữ, backup tốt
   2. intfloat/multilingual-e5-small  —  Nhẹ hơn e5-base ~3x, nhanh hơn

▶  Gọi run_all_benchmarks() ở cell tiếp theo để chạy.


In [14]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 4b — Chạy toàn bộ benchmark + bảng so sánh
# Chạy cell này sau Cell 3 (corpus đã build xong)
# ═══════════════════════════════════════════════════════════════════════════
import pandas as pd

# ── Chạy từng model ──────────────────────────────────────────────────────────
all_results = []
for cfg in MODEL_CONFIGS:
    result = run_model_pipeline(
        cfg,
        chunks    = all_chunks,
        qa_pairs  = val_qa if val_qa else train_qa,   # ưu tiên val set
        k_list    = (1, 3, 5, 10),
        eval_sample = 500,
    )
    all_results.append(result)

# ── Bảng so sánh ─────────────────────────────────────────────────────────────
print(f'\n{"═"*80}')
print('🏆  BENCHMARK SUMMARY')
print(f'{"═"*80}')

df = pd.DataFrame(all_results).sort_values('recall@5', ascending=False)
df['mrr_%']      = (df['mrr'] * 100).round(1)
df['recall@1_%'] = (df['recall@1'] * 100).round(1)
df['recall@5_%'] = (df['recall@5'] * 100).round(1)
df['recall@10_%']= (df['recall@10'] * 100).round(1)

display_cols = ['model', 'dim', 'encode_time_s', 'mrr_%',
                'recall@1_%', 'recall@5_%', 'recall@10_%', 'note']
print(df[display_cols].to_string(index=False))

# ── Chọn model tốt nhất theo Recall@5 ────────────────────────────────────────
best = df.iloc[0]
print(f'\n🥇 Best model (Recall@5): {best["model"]}')
print(f'   Recall@1={best["recall@1_%"]}%  '
      f'Recall@5={best["recall@5_%"]}%  '
      f'MRR={best["mrr_%"]}%  '
      f'encode={best["encode_time_s"]}s')

# ── Tự động set model tốt nhất làm active model cho các cell sau ─────────────
BEST_CFG = next(c for c in MODEL_CONFIGS if c['name'] == best['model'])
print(f'\n✅ Đặt MODEL_NAME = "{BEST_CFG["name"]}" cho pipeline downstream.')

# Các biến global mà Cell 7/8/9 cần — load lại từ best model
MODEL_NAME     = BEST_CFG['name']
QUERY_PREFIX   = BEST_CFG['query_prefix']
PASSAGE_PREFIX = BEST_CFG['passage_prefix']

# Re-run pipeline với best model để lấy embed_model + index cho Cell 7/8/9
print('\n⏳ Re-loading best model cho downstream cells...')
embed_model = SentenceTransformer(MODEL_NAME, device=device, cache_folder=MODEL_CACHE)
DIM         = embed_model.get_sentence_embedding_dimension()

texts      = [PASSAGE_PREFIX + c['context'] for c in all_chunks]
embeddings = embed_model.encode(
    texts, batch_size=BATCH_SIZE,
    normalize_embeddings=True, show_progress_bar=True, convert_to_numpy=True,
).astype(np.float32)

cpu_index  = faiss.IndexFlatIP(DIM)
cpu_index.add(embeddings)
if FAISS_GPU:
    try:
        res   = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
    except:
        index = cpu_index
else:
    index = cpu_index

print(f'✅ Pipeline ready — Cell 5/6/7/8/9 có thể chạy bình thường.')


════════════════════════════════════════════════════════════════════
🔄  intfloat/multilingual-e5-base
    note: Đa ngôn ngữ, backup tốt
════════════════════════════════════════════════════════════════════
✅  Loaded  |  dim=768  max_seq=512


Batches:   0%|          | 0/292 [00:00<?, ?it/s]

KeyboardInterrupt: 

## ⚡ Cell 5 — Encode toàn bộ Corpus (batch GPU)

GPU encode **~200–300 passages/giây** — ViQuAD2 (~5k–20k chunks) xong trong 5–10 phút.

In [24]:
import numpy as np
import time

# Tự động chọn batch_size theo VRAM
if torch.cuda.is_available():
    vram_gb    = torch.cuda.get_device_properties(0).total_memory / 1e9
    BATCH_SIZE = 128 if vram_gb >= 8 else 64
else:
    BATCH_SIZE = 16

# Prefix theo convention của bi-encoder
PASSAGE_PREFIX = 'passage: '
QUERY_PREFIX   = 'query: '   # dùng khi encode câu hỏi lúc retrieve

texts = [PASSAGE_PREFIX + c['context'] for c in all_chunks]

print(f'⏳ Encoding {len(texts):,} passages (batch_size={BATCH_SIZE}, device={device.upper()})...')
t0 = time.time()

embeddings = embed_model.encode(
    texts,
    batch_size=BATCH_SIZE,
    normalize_embeddings=True,  # L2-normalize → cosine similarity = dot product
    show_progress_bar=True,
    convert_to_numpy=True,
)

elapsed = time.time() - t0
print(f'\n✅ Done!')
print(f'   Shape   : {embeddings.shape}')
print(f'   Dtype   : {embeddings.dtype}')
print(f'   Time    : {elapsed:.1f}s  ({len(texts)/elapsed:.0f} passages/s)')
print(f'   RAM     : {embeddings.nbytes / 1e6:.1f} MB')

⏳ Encoding 4,658 passages (batch_size=128, device=CUDA)...


Batches:   0%|          | 0/37 [00:00<?, ?it/s]


✅ Done!
   Shape   : (4658, 768)
   Dtype   : float32
   Time    : 69.4s  (67 passages/s)
   RAM     : 14.3 MB


## 🗂️ Cell 6 — Build FAISS Index

- **IndexFlatIP**: exact cosine, dùng khi N < 100k (phù hợp ViQuAD2)
- **IndexIVFFlat**: approximate, tự động chọn nếu corpus lớn hơn

In [16]:
import faiss
import numpy as np

N    = len(all_chunks)
vecs = embeddings.astype(np.float32)

print(f'Building FAISS index: {N:,} vectors, dim={DIM}')

if N < 100_000:
    cpu_index  = faiss.IndexFlatIP(DIM)
    index_type = 'IndexFlatIP'
else:
    nlist      = min(int(N ** 0.5), 512)
    quantizer  = faiss.IndexFlatIP(DIM)
    cpu_index  = faiss.IndexIVFFlat(quantizer, DIM, nlist, faiss.METRIC_INNER_PRODUCT)
    index_type = f'IndexIVFFlat(nlist={nlist})'
    cpu_index.train(vecs)

cpu_index.add(vecs)

# Chuyển lên GPU nếu có
if FAISS_GPU:
    try:
        res   = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
        print(f'✅ FAISS index on GPU  ({index_type})')
    except Exception as e:
        index = cpu_index
        print(f'⚠️  GPU transfer failed, dùng CPU: {e}')
else:
    index = cpu_index
    print(f'✅ FAISS index on CPU  ({index_type})')

print(f'   ntotal : {index.ntotal:,}')

# Sanity check
scores, idxs = index.search(vecs[0:1], 1)
print(f'   Self-query score: {scores[0][0]:.6f}  ← phải = 1.000000')

Building FAISS index: 4,658 vectors, dim=768
✅ FAISS index on CPU  (IndexFlatIP)
   ntotal : 4,658
   Self-query score: 1.000000  ← phải = 1.000000


## 🔍 Cell 7 — Test Retrieval

In [25]:
def retrieve(query: str, top_k: int = 5, verbose: bool = True) -> list:
    """Retrieve top-k passages cho một câu hỏi tiếng Việt."""
    q_vec = embed_model.encode(
        [QUERY_PREFIX + query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    scores, indices = index.search(q_vec, top_k)

    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
        if idx == -1:
            continue
        chunk = all_chunks[int(idx)]
        results.append({
            'rank'    : rank + 1,
            'score'   : float(score),
            'chunk_id': int(idx),
            'title'   : chunk['title'],
            'context' : chunk['context'],
        })

    if verbose:
        print(f'\n🔍 Query: "{query}"')
        print('─' * 72)
        for r in results:
            print(f'  [Rank {r["rank"]}] score={r["score"]:.4f} | {r["title"]}')
            print(f'           {r["context"][:200]}...')
            print()

    return results


# Test với 3 câu hỏi ngẫu nhiên từ tập train
import random
random.seed(42)
test_qs = random.sample([q for q in train_qa if not q['is_impossible']], 3)

for qa in test_qs:
    results = retrieve(qa['question'], top_k=3)
    gt_id   = qa['chunk_id']
    hit     = any(r['chunk_id'] == gt_id for r in results)
    status  = '✅ HIT' if hit else '❌ MISS'
    print(f'  {status}  (ground-truth chunk_id={gt_id})\n')


🔍 Query: "Bên cạnh Kinh Châu, tại nơi nào cũng nổ ra nhiều cuộc kháng chiến của các bộ tộc thiểu số?"
────────────────────────────────────────────────────────────────────────
  [Rank 1] score=0.8240 | Nhà Hán
           Năm 132, Thuận Đế lấy vợ là Thuận Liệt Lương Hoàng hậu, từ đó họ Lương (梁) bắt đầu tham gia triều chính. Họ Lương có nguồn gốc từ Lương Nhiễu, làm Thái thú quận Tửu Tuyền thời Vương Mãng, cùng Đậu Dun...

  [Rank 2] score=0.8206 | Trung Hoa Dân Quốc (1912-1949)
           Sau khi Chính phủ Quốc Dân định đô tại Nam Kinh năm 1927, phế bỏ cấp đạo, lập riêng khu đốc sát hành chính, đổi tên hai tỉnh Trực Lệ và Phụng Thiên là Hà Bắc và Liêu Ninh, nhập địa phương Kinh Triệu v...

  [Rank 3] score=0.8146 | Hà Nam (Trung Quốc)
           Do ảnh hưởng từ khởi nghĩa Vũ Xương, đảng viên cách mạng Hà Nam đã thúc đẩy Tân quân tại tỉnh lỵ Khai Phong phản lại chính quyền, tổ chức khởi nghĩa vũ trang ở các phủ huyện khác, song không giành đượ...

  ✅ HIT  (ground-truth chunk_id=788)




## 📊 Cell 8 — Đánh giá Retrieval (Recall@K)

Kiểm tra xem FAISS có retrieve đúng context cho mỗi câu hỏi không.

In [26]:
def evaluate_retrieval(qa_pairs, k_list=(1, 3, 5, 10),
                        sample_size=500, skip_impossible=True):
    """Tính Recall@K trên tập QA pairs."""
    pool = [q for q in qa_pairs if not q['is_impossible']] if skip_impossible else qa_pairs
    if len(pool) > sample_size:
        random.seed(42)
        pool = random.sample(pool, sample_size)

    max_k = max(k_list)
    hits  = {k: 0 for k in k_list}

    print(f'⏳ Evaluating {len(pool):,} QA pairs (skip_impossible={skip_impossible})...')

    for qa in tqdm(pool, desc='Recall@K'):
        gt_id = qa['chunk_id']

        q_vec = embed_model.encode(
            [QUERY_PREFIX + qa['question']],
            normalize_embeddings=True,
            convert_to_numpy=True,
        ).astype(np.float32)

        _, indices = index.search(q_vec, max_k)
        retrieved  = [int(i) for i in indices[0] if i != -1]

        for k in k_list:
            if gt_id in retrieved[:k]:
                hits[k] += 1

    n = len(pool)
    print(f'\n📊 Retrieval Evaluation — model: {MODEL_NAME}')
    print(f'   Sample: {n:,} pairs | skip_impossible={skip_impossible}')
    print()
    for k in k_list:
        r   = hits[k] / n * 100
        bar = '█' * int(r / 5) + '░' * (20 - int(r / 5))
        tip = '✅' if r >= 85 else ('⚠️' if r >= 70 else '❌')
        print(f'   Recall@{k:<3}: {r:5.1f}%  |{bar}| {tip}')

    return {k: round(hits[k] / n, 4) for k in k_list}


recall_scores = evaluate_retrieval(train_qa, sample_size=500)

⏳ Evaluating 500 QA pairs (skip_impossible=True)...


Recall@K:   0%|          | 0/500 [00:00<?, ?it/s]


📊 Retrieval Evaluation — model: intfloat/multilingual-e5-base
   Sample: 500 pairs | skip_impossible=True

   Recall@1  :  65.2%  |█████████████░░░░░░░| ❌
   Recall@3  :  80.6%  |████████████████░░░░| ⚠️
   Recall@5  :  87.6%  |█████████████████░░░| ✅
   Recall@10 :  91.6%  |██████████████████░░| ✅


## 💾 Cell 9 — Export Artifacts

Lưu tất cả artifacts cần thiết để chạy RAG local.

In [61]:
import json

print(f'💾 Saving to: {DATA_DIR}\n')

# 1. FAISS index
faiss_path = os.path.join(DATA_DIR, 'faiss_index.bin')
faiss.write_index(index, faiss_path)
print(f'  ✅ faiss_index.bin   {os.path.getsize(faiss_path)/1e6:.1f} MB')

# 2. Chunks (contexts + metadata)
chunks_path = os.path.join(DATA_DIR, 'chunks.json')
with open(chunks_path, 'w', encoding='utf-8') as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)
print(f'  ✅ chunks.json       {os.path.getsize(chunks_path)/1e6:.1f} MB  ({len(all_chunks):,} chunks)')

# 3. QA pairs train + val (để eval end-to-end)
qa_path = os.path.join(DATA_DIR, 'qa_pairs.json')
all_qa  = train_qa + val_qa
with open(qa_path, 'w', encoding='utf-8') as f:
    json.dump(all_qa, f, ensure_ascii=False, indent=2)
print(f'  ✅ qa_pairs.json     {os.path.getsize(qa_path)/1e6:.1f} MB  ({len(all_qa):,} pairs)')

# 4. Config (để local pipeline biết load đúng model + params)
config = {
    'embed_model'    : MODEL_NAME,
    'embed_dim'      : int(DIM),
    'index_type'     : index_type,
    'n_chunks'       : len(all_chunks),
    'n_qa_train'     : len(train_qa),
    'n_qa_val'       : len(val_qa),
    'query_prefix'   : QUERY_PREFIX,
    'passage_prefix' : PASSAGE_PREFIX,
    'normalize'      : True,
    'recall_scores'  : {f'recall@{k}': v for k, v in recall_scores.items()},
}
config_path = os.path.join(DATA_DIR, 'config.json')
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
print(f'  ✅ config.json')

# 5. Raw embeddings (optional — dùng để fine-tune hoặc debug)
emb_path = os.path.join(DATA_DIR, 'embeddings.npy')
np.save(emb_path, embeddings)
print(f'  ✅ embeddings.npy    {embeddings.nbytes/1e6:.1f} MB')

print(f'\n🎉 Xong! Artifacts sẵn sàng để dùng trong local pipeline.')
print(f'   → Dùng config.json để load đúng model khi chạy src/retriever.py')

💾 Saving to: /content/drive/MyDrive/Code/Colab/data

  ✅ faiss_index.bin   14.3 MB
  ✅ chunks.json       5.5 MB  (4,658 chunks)
  ✅ qa_pairs.json     49.5 MB  (32,268 pairs)
  ✅ config.json
  ✅ embeddings.npy    14.3 MB

🎉 Xong! Artifacts sẵn sàng để dùng trong local pipeline.
   → Dùng config.json để load đúng model khi chạy src/retriever.py


## 🔎 Cell 12 — [Optional] Phân tích chất lượng retrieval

Xem các câu hỏi mà retriever fail để debug.

In [ ]:
import random

def debug_failures(qa_pairs, top_k=5, n_failures=5):
    """In ra các câu hỏi mà retriever không tìm được đúng context."""
    failures = []
    sample = random.sample([q for q in qa_pairs if not q['is_impossible']], 200)

    for qa in sample:
        gt_id = qa['chunk_id']
        q_vec = embed_model.encode(
            ["query: " + qa['question']],
            normalize_embeddings=True,
            convert_to_numpy=True,
        ).astype(np.float32)

        scores, indices = gpu_index.search(q_vec, top_k)
        retrieved_ids = [all_chunks[i]['chunk_id'] for i in indices[0] if i != -1]

        if gt_id not in retrieved_ids:
            failures.append({
                'question'       : qa['question'],
                'gt_context'     : all_chunks[gt_id]['context'][:300],
                'top1_context'   : all_chunks[indices[0][0]]['context'][:300],
                'top1_score'     : float(scores[0][0]),
            })
        if len(failures) >= n_failures:
            break

    print(f"📋 {len(failures)} failure examples (out of 200 sampled):")
    for i, f in enumerate(failures):
        print(f"\n--- Failure #{i+1} ---")
        print(f"Question  : {f['question']}")
        print(f"GT context: {f['gt_context']}...")
        print(f"Top-1 retrieved (score={f['top1_score']:.4f}):")
        print(f"  {f['top1_context']}...")


debug_failures(train_qa)

---

## ✅ Checklist sau khi chạy xong

```
~/projects/rag-viquad/
├── .hf_cache/                ← HuggingFace dataset cache  (gitignore)
├── .model_cache/             ← PhoBERT weights            (gitignore)
├── data/
│   ├── faiss_index.bin       ← load bằng: faiss.read_index('data/faiss_index.bin')
│   ├── chunks.json           ← list context + metadata
│   ├── qa_pairs.json         ← train + val QA để eval end-to-end
│   ├── config.json           ← model name, dim, recall scores
│   └── embeddings.npy        ← raw embeddings (optional)
├── notebooks/
│   └── ViQuAD2_RAG_Colab.ipynb
└── src/                      ← bước tiếp: retriever.py, reader.py, pipeline.py
```

**Load lại trên local pipeline:**

```python
import faiss, json

index  = faiss.read_index('data/faiss_index.bin')
chunks = json.load(open('data/chunks.json',  encoding='utf-8'))
config = json.load(open('data/config.json',  encoding='utf-8'))

print(config['recall_scores'])  # {'recall@1': 0.xx, 'recall@5': 0.xx, ...}
```

**Recommend `.gitignore`:**

```
.hf_cache/
.model_cache/
data/embeddings.npy
data/faiss_index.bin
```